# Yogera — train a Luganda adapter on a free T4 (no GPU of your own)

**Runtime → Change runtime type → T4 GPU.** Everything here runs on the free tier;
~1–2 hours of wall clock for a few thousand short conversations.

Reality check before you start:
- A T4 (16 GB, fp16 only, no bf16) **cannot** hold Sunflower-14B for training. This
  notebook uses **Qwen/Qwen3-8B** at 4-bit with 1024-token sequences — the biggest thing
  that fits comfortably. It knows much less Luganda than Sunflower, so expect a model
  that copies your store's voice well and still makes agreement mistakes.
- **Data beats hardware.** 2,000 native-checked turns on an 8B model will beat 200,000
  scraped turns on a 32B model. Do the review first.
- A frontier model + `app/prompts.py` + the retrieval memory in `app/knowledge.py`
  is still better Luganda today. This notebook is how you stop paying for it, not how
  you beat it on day one.

## 1. Get the code and the training stack.

Upload your project (or `git clone` it) so `/content/yogera` exists.

In [ ]:
# Option A: from a git remote
# !git clone https://github.com/<you>/yogera.git /content/yogera
# Option B: from a zip you uploaded to /content
# !unzip -q /content/yogera-handoff.zip -d /content/

import os, torch
os.chdir('/content/yogera')
assert os.path.exists('finetune/build_dataset.py'), 'the repo is not in /content/yogera'
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!pip -q install "torch" "transformers>=4.44" "peft>=0.11" "trl>=0.9" "datasets>=2.19" \
    "accelerate>=0.30" "bitsandbytes>=0.43" "pyyaml" "hf_transfer"
import transformers, peft, trl, bitsandbytes
print('transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__)

## 2. Build the dataset.

Put your own Luganda conversations in `finetune/data/ingested/` (see
`finetune/ingest_hf.py`) and/or chat with the app and export. Then:

In [ ]:
!python finetune/validate_dataset.py finetune/data | tail -20
!python finetune/build_dataset.py

In [ ]:
import json
stats = json.load(open('finetune/out/stats.json'))
print(json.dumps({k: stats[k] for k in ('records','words_total','duplicates_removed','verified_fraction')}, indent=2))
assert stats['records']['total'] > 100, 'Too few rows. Add a corpus before training.'

## 3. Check the plan (10 seconds, no training).

In [ ]:
!python finetune/train_lora.py --config finetune/configs/lora_t4_8b.yaml --dry-run

## 4. Train. This is the long cell — ~1–2 h on a T4. Keep the tab open.

In [ ]:
!python finetune/train_lora.py --config finetune/configs/lora_t4_8b.yaml

## 5. Score it before you believe it.

`eval.py` needs an API key to *generate* answers. Without one, upload a predictions
file (`{"id":..., "prediction":...}` per line) from your own inference and pass
`--predictions`. Either way you get chrF++ and a `human_rubric.csv` to hand to a
native speaker.

In [ ]:
# generate with the model you just trained (after step 6 serving), or:
# !python finetune/eval.py --predictions /content/preds.jsonl
!ls -la finetune/out/human_rubric.csv 2>/dev/null || echo 'run eval.py first'

## 6. Take it home: GGUF for Ollama / llama.cpp.

Two conversions: merge the adapter into fp16 weights (already done if
`merge_and_export: true`), then quantise to Q4_K_M (~5 GB, runs on a laptop CPU).

In [ ]:
import os, glob
merged = 'finetune/out/yogera-t4-8b-merged'
print('merged weights:', len(glob.glob(merged + '/*.safetensors')), 'files')
if not os.path.isdir(merged):
    raise SystemExit('run training with merge_and_export: true first')

In [ ]:
!git clone -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt
!cd /content/llama.cpp && python convert_hf_to_gguf.py /content/yogera/finetune/out/yogera-t4-8b-merged \
    --outfile /content/yogera-yogera-t4-8b-f16.gguf --outtype f16

In [ ]:
# quantise (build only what we need, ~3 min on the free CPU)
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF > /dev/null && cmake --build build --config Release -j --target llama-quantize > /dev/null
!ls -lh /content/yogera-yogera-t4-8b-f16.gguf
!cd /content/llama.cpp && ./build/bin/llama-quantize /content/yogera-yogera-t4-8b-f16.gguf /content/yogera-Q4_K_M.gguf Q4_K_M

In [ ]:
from google.colab import files
files.download('/content/yogera-Q4_K_M.gguf')   # then serve it locally, see below

## 7. Serve it locally, point the app at it

```bash
# on your own machine, once
cat > Modelfile <<'EOF'
FROM ./yogera-Q4_K_M.gguf
PARAMETER temperature 0.6
PARAMETER num_ctx 4096
SYSTEM "Oli Yogera, omuyambi ayogera Oluganda."
EOF
ollama create yogera-luganda -f Modelfile
ollama serve
```

```env
# .env in the Yogera project
LLM_PROVIDER=compatible
LLM_BASE_URL=http://localhost:11434/v1
LLM_MODEL=yogera-luganda
LLM_API_KEY=ollama
```

Then open the app: it is now running your fine-tune, locally, with no API bill.

## Cleanup
Colab disconnects. Save `finetune/out/*.jsonl` (dataset) and the `.gguf` to Drive
before the session ends, or you will redo an hour of work.